# 03 - Cohort Retention & Churn

This notebook groups customers into acquisition cohorts, builds a retention matrix, and models time-to-churn using survival analysis (Kaplan-Meier, Cox proportional hazards) — the same methodology used for time-to-event outcomes in a clinical setting, applied here to customer behaviour.

## Research Questions

1. Does retention differ by acquisition cohort, country, or first-order value?
2. How do we define churn on non-contractual data, where there is no explicit cancellation-of-relationship event?
3. Which customer characteristics are associated with faster churn?

## Plan

0. Setup & Loading
1. Cohort Assignment
2. Retention Matrix
3. Decision — Defining Churn
4. Kaplan-Meier Survival Curves
5. Cox Proportional Hazards Model
6. Export


# 0. Setup & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from pathlib import Path
from IPython.display import display, Markdown

PROC = Path("../data/processed")
clean = pd.read_parquet(PROC / "clean_transactions.parquet")
segments = pd.read_parquet(PROC / "customer_segments.parquet")

REFERENCE_DATE = clean["InvoiceDate"].max() + pd.Timedelta(1, unit="D")

# Non-cancelled purchases only: a cancelled invoice is not a purchase event
purchases = clean[~clean["IsCancellation"]].copy()

display(Markdown(f"""
### Input
* **Purchase rows (excluding cancellations):** `{len(purchases):,}`
* **Unique customers:** `{purchases['Customer ID'].nunique():,}`
"""))


**Results.** 803,001 purchase rows, 5,854 unique customers (slightly fewer than the 5,882 in notebook 01, since a handful of customers had only cancellations and no genuine purchase — consistent with the `Frequency = 0` customers already identified and excluded in notebook 02).

# 1. Cohort Assignment

A customer's cohort is the calendar month of their first purchase.

In [ ]:
first_purchase = purchases.groupby("Customer ID")["InvoiceDate"].min().rename("FirstPurchaseDate")
purchases = purchases.merge(first_purchase, on="Customer ID")

purchases["CohortMonth"] = purchases["FirstPurchaseDate"].dt.to_period("M")
purchases["InvoiceMonth"] = purchases["InvoiceDate"].dt.to_period("M")
purchases["MonthsSinceAcquisition"] = (
    (purchases["InvoiceMonth"] - purchases["CohortMonth"]).apply(lambda x: x.n)
)

cohort_sizes = purchases.groupby("CohortMonth")["Customer ID"].nunique()
display(Markdown("### Cohort sizes (customers acquired per month)"))
display(cohort_sizes)


**Results.** Cohort sizes range from 951 (Dec. 2009, the launch month — includes all pre-existing customers by construction) down to 28 (Dec. 2011, the final partial month). Most monthly cohorts settle between 100 and 400 customers. The size asymmetry between early and late cohorts matters for section 2: late cohorts (e.g. Nov./Dec. 2011) can only be observed for a few weeks, which the survival analysis in sections 4-5 handles correctly via censoring, but the retention matrix below should be read with this in mind — later cohorts will structurally have fewer populated columns.

# 2. Retention Matrix

For each cohort, what share of its customers made at least one purchase in each subsequent month?

In [ ]:
cohort_activity = (
    purchases.groupby(["CohortMonth", "MonthsSinceAcquisition"])["Customer ID"]
    .nunique()
    .reset_index()
)

cohort_pivot = cohort_activity.pivot(index="CohortMonth", columns="MonthsSinceAcquisition", values="Customer ID")
retention_matrix = cohort_pivot.divide(cohort_pivot[0], axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(retention_matrix, annot=False, cmap="Blues", vmin=0, vmax=100, ax=ax)
ax.set_title("Retention rate (%) by cohort and months since acquisition")
plt.tight_layout()
plt.show()

display(Markdown("### Retention matrix (first 6 months)"))
display(retention_matrix.iloc[:, :7].round(1))


**Results.** Month-1 retention starts around 35% for the earliest cohort (Dec. 2009) and drops for most subsequent cohorts, often into the 15-25% range — a steep early drop-off is typical for non-contractual retail and is not itself a problem. More notable: cohorts acquired from **Sep.-Dec. 2010 show visibly weaker retention** (month-1 retention around 9-24%, compared to 20-35% for cohorts earlier in the year). This could reflect seasonal, one-off Christmas shoppers who never return, rather than a genuine decline in cohort quality — worth keeping in mind rather than concluding the business was "getting worse" at acquiring customers.

# 3. Decision — Defining Churn

There is no explicit "cancel my account" event in this dataset — customers simply stop purchasing. Churn must therefore be defined via an inactivity threshold, chosen from the data.

In [ ]:
# Inter-purchase interval: gap between consecutive orders, customers with 2+ purchases only.
# Customers with a single purchase are filtered out BEFORE computing gaps: applying .diff()
# to a single-date series returns an empty Series, and pandas' .apply() does not reliably
# preserve empty-Series results across a mixed-length groupby, causing a TypeError in pd.concat.
purchase_counts = purchases.groupby("Customer ID")["InvoiceDate"].nunique()
multi_purchase_customers = purchase_counts[purchase_counts >= 2].index
multi = purchases[purchases["Customer ID"].isin(multi_purchase_customers)]

def compute_gaps(dates):
    d = pd.Series(sorted(dates.unique()))
    return d.diff().dt.days.dropna()

all_gaps = multi.groupby("Customer ID")["InvoiceDate"].apply(compute_gaps).reset_index(drop=True)

display(Markdown(f"**Customers with 2+ purchases:** `{len(multi_purchase_customers):,}` / `{purchase_counts.shape[0]:,}`"))
display(Markdown("### Distribution of inter-purchase gaps (days)"))
display(all_gaps.describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(all_gaps[all_gaps < 365], bins=50, ax=ax)
ax.set_title("Inter-purchase gaps under 365 days")
plt.show()

for pct in [75, 90, 95]:
    display(Markdown(f"* **{pct}th percentile:** `{all_gaps.quantile(pct/100):.0f}` days"))


**Results.** 4,235 of 5,854 customers (72%) made 2+ purchases. Gap distribution: median 25 days, 75th percentile 62 days, 90th percentile 136 days, 95th percentile 207 days — heavily right-skewed (mean 51.8 vs. median 25), consistent with a mix of frequent buyers and occasional repeat customers.

**Decision — churn threshold.** Set at **180 days**, between the 90th (136) and 95th (207) percentiles: long enough that a customer crossing it has genuinely gone quiet by the standards of this customer base (only ~7-10% of real repeat purchases take this long), while not so long that almost every customer looks "still active." This sits deliberately between the two most common conventions (90th/95th percentile) rather than defaulting to a round number like 90 or 365 days without checking it against this distribution.

In [ ]:
CHURN_THRESHOLD_DAYS = 180

customer_survival = purchases.groupby("Customer ID").agg(
    FirstPurchaseDate=("FirstPurchaseDate", "first"),
    LastPurchaseDate=("InvoiceDate", "max"),
    Country=("Country", "first"),
    NPurchases=("Invoice", "nunique"),
).reset_index()

customer_survival["DurationDays"] = (
    customer_survival["LastPurchaseDate"] - customer_survival["FirstPurchaseDate"]
).dt.days

# Right-censoring: a customer whose last purchase is recent enough (within the
# churn threshold of the reference date) has NOT necessarily churned — we simply
# have not observed the outcome yet. event=1 means churn was observed; event=0
# means the customer is censored (still possibly active).
days_since_last = (REFERENCE_DATE - customer_survival["LastPurchaseDate"]).dt.days
customer_survival["Event"] = (days_since_last > CHURN_THRESHOLD_DAYS).astype(int)

# For censored customers, duration is measured up to the reference date, not
# up to their last purchase, since we have not observed them churn yet.
customer_survival["Duration"] = np.where(
    customer_survival["Event"] == 1,
    customer_survival["DurationDays"],
    (REFERENCE_DATE - customer_survival["FirstPurchaseDate"]).dt.days
)

display(Markdown(f"**Observed churn events:** `{customer_survival['Event'].sum():,}` / `{len(customer_survival):,}` (`{customer_survival['Event'].mean()*100:.1f}%`)"))
display(customer_survival[["Duration", "Event"]].describe())


**Results.** 2,380 / 5,854 customers (40.7%) are observed to churn under this threshold; the remaining 59.3% are censored — meaning we simply haven't observed them long enough (or recently enough) to know if they've truly stopped, not that they are "loyal forever." This is the same censoring logic as METABRIC's overall-survival variable. A 40.7% event rate is workable for Kaplan-Meier, though — as shown below — it does mean the median survival time is not reached within the observation window.

# 4. Kaplan-Meier Survival Curves

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(customer_survival["Duration"], event_observed=customer_survival["Event"])

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title("Overall customer retention (Kaplan-Meier)")
ax.set_xlabel("Days since first purchase")
ax.set_ylabel("Retention probability")
plt.show()

display(Markdown(f"**Median survival time:** `{kmf.median_survival_time_}` days"))


**Median survival time is undefined (`inf`).** With only 40.7% of customers observed to churn under the 180-day threshold, the Kaplan-Meier curve never crosses 50% survival within the observation window — this is expected, not an error. It means most customers have not (yet) crossed the inactivity threshold by the end of the dataset. The median is not a usable summary statistic here; survival probabilities at fixed horizons (e.g. "% still active at 12 months") are the right way to report this curve instead.

In [ ]:
# Stratified by first-order value tier: does spending more at first purchase
# predict staying longer?
first_order_value = purchases.sort_values("InvoiceDate").groupby("Customer ID").first()["LineRevenue"]
customer_survival = customer_survival.merge(
    first_order_value.rename("FirstOrderValue"), on="Customer ID"
)
customer_survival["ValueTier"] = pd.qcut(
    customer_survival["FirstOrderValue"], 3, labels=["Low", "Medium", "High"]
)

fig, ax = plt.subplots(figsize=(8, 5))
for tier in ["Low", "Medium", "High"]:
    mask = customer_survival["ValueTier"] == tier
    kmf_tier = KaplanMeierFitter()
    kmf_tier.fit(
        customer_survival.loc[mask, "Duration"],
        event_observed=customer_survival.loc[mask, "Event"],
        label=tier,
    )
    kmf_tier.plot_survival_function(ax=ax)
ax.set_title("Retention by first-order value tier")
plt.show()

groups = customer_survival["ValueTier"]
results = logrank_test(
    customer_survival.loc[groups == "Low", "Duration"], customer_survival.loc[groups == "High", "Duration"],
    customer_survival.loc[groups == "Low", "Event"], customer_survival.loc[groups == "High", "Event"],
)
display(Markdown(f"**Log-rank test (Low vs. High tier), p-value:** `{results.p_value:.4f}`"))


**Results.** Log-rank test p = 0.7898 — **not statistically significant**. First-order value does not predict time-to-churn in this dataset. This is a genuine, reportable finding, not a failed analysis: the working hypothesis ("customers who spend more upfront stay longer") is not supported by the data, and this is stated plainly rather than reframed to sound more positive.

# 5. Cox Proportional Hazards Model

Kaplan-Meier compares groups one variable at a time. Cox regression estimates the effect of several covariates simultaneously, each expressed as a hazard ratio — but only once the proportional-hazards assumption is checked.

In [ ]:
cox_data = customer_survival[["Duration", "Event", "FirstOrderValue", "NPurchases", "Country"]].copy()

# Keep the top countries only; group the long tail to avoid an unstable model
# with dozens of near-empty dummy categories.
top_countries = cox_data["Country"].value_counts().nlargest(5).index
cox_data["Country"] = cox_data["Country"].where(cox_data["Country"].isin(top_countries), "Other")
cox_data = pd.get_dummies(cox_data, columns=["Country"], drop_first=True)

cph = CoxPHFitter()
cph.fit(cox_data, duration_col="Duration", event_col="Event")

display(Markdown("### Cox model summary (unstratified — see assumption check below)"))
display(cph.summary.round(4))


In [ ]:
cph.check_assumptions(cox_data, p_value_threshold=0.05, show_plots=False)


**Results.** `NPurchases` **fails** the proportional-hazards test (p < 5e-05): its effect on churn risk is not constant over time, so its hazard ratio in the unstratified model above cannot be trusted at face value. The standard correction is to stratify by this variable (letting the baseline hazard vary freely across purchase-frequency groups) rather than forcing it into a single invalid coefficient.

In [ ]:
cox_data["PurchaseBin"] = pd.qcut(cox_data["NPurchases"], 4, labels=False, duplicates="drop")

cph_stratified = CoxPHFitter()
cph_stratified.fit(
    cox_data.drop(columns=["NPurchases"]),
    duration_col="Duration",
    event_col="Event",
    strata=["PurchaseBin"],
)

display(Markdown("### Cox model, stratified by NPurchases quartile"))
display(cph_stratified.summary.round(4))

cph_stratified.check_assumptions(cox_data.drop(columns=["NPurchases"]), p_value_threshold=0.05, show_plots=False)


**Results — final interpretation.** After stratifying on purchase-frequency quartile, the proportional-hazards assumption holds ("Proportional hazard assumption looks okay"). **No remaining covariate is statistically significant**: every p-value exceeds 0.05, and every confidence interval for `exp(coef)` includes 1. `Country_Germany` comes closest (p = 0.080, HR = 0.526, suggesting possibly better retention) but does not cross the conventional significance threshold.

This is consistent with the non-significant log-rank test on first-order value tiers above (p = 0.79): **purchase frequency is the dominant, genuine signal in this dataset** for time-to-churn. First-order value and country do not add detectable predictive power once frequency is properly accounted for. This is reported as a real finding, not a modelling failure — a model that correctly separates a strong signal (frequency, handled via stratification) from non-signal (country, first-order value) is more trustworthy than one where everything appears "significant."

**Practical implication.** A retention strategy targeting early purchase-frequency behaviour (e.g. encouraging a fast second and third purchase) is better supported by this data than one segmented by acquisition country or first-order basket size.

# 6. Export

In [ ]:
cols = ["Customer ID", "Duration", "Event", "FirstOrderValue", "NPurchases", "Country", "ValueTier"]
customer_survival[cols].to_parquet(PROC / "customer_survival.parquet", index=False)
display(Markdown(f"Saved `{len(customer_survival):,}` customer survival records to `customer_survival.parquet`"))
